In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [31]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("train shape: ", train.shape)
print("test shape: " , test.shape)

train shape:  (577347, 12)
test shape:  (247435, 11)


In [32]:
train.head()

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
0,0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,GALAXY
1,1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,GALAXY
2,2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,QSO
3,3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,GALAXY
4,4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,GALAXY


In [33]:
train.describe()


,id,alpha,delta,u,g,r,i,z,redshift
count,577347.00000,577347.000000,577347.000000,577347.000000,577347.000000,577347.000000,577347.000000,577347.000000,577347.000000
mean,288673.00000,181.616673,21.834654,22.441926,21.007273,19.962811,19.378911,19.041136,0.723135
std,166665.86727,96.242941,18.933570,2.018135,1.795426,1.648964,1.580059,1.584365,0.810070
min,0.00000,0.011684,-17.966988,-0.139225,13.535483,12.579407,11.962781,11.682803,-0.009970
25%,144336.50000,132.161499,2.474097,20.977090,19.865005,18.820671,18.306820,17.973192,0.181052
50%,288673.00000,188.681465,21.484412,22.570222,21.467820,20.431153,19.631642,19.188598,0.497525
75%,433009.50000,231.829693,36.988310,23.869103,22.292715,21.164096,20.608191,20.162111,0.881390
max,577346.00000,359.999810,79.158322,28.253263,27.620208,25.254499,27.910853,26.826867,7.010780


In [34]:
# Missing values
print(train.isna().sum())

id                   0
alpha                0
delta                0
u                    0
g                    0
r                    0
i                    0
z                    0
redshift             0
spectral_type        0
galaxy_population    0
class                0
dtype: int64


In [35]:
print(test.isna().sum())

id                   0
alpha                0
delta                0
u                    0
g                    0
r                    0
i                    0
z                    0
redshift             0
spectral_type        0
galaxy_population    0
dtype: int64


In [36]:
numeric_cols = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']

for col in numeric_cols:
    Q1 = train[col].quantile(0.25)
    Q3 = train[col].quantile(0.75)
    IQR = Q3 - Q1

    outliers = train[(train[col] < Q1 - 1.5 * IQR) | (train[col] > Q3 - 1.5 * IQR)]


    print(f"{col}: {len(outliers)} outliers")

alpha: 482444 outliers
delta: 577166 outliers
u: 529066 outliers
g: 492902 outliers
r: 499774 outliers
i: 507667 outliers
z: 506921 outliers
redshift: 577347 outliers


the outliers are a great percentage of the data but that is not a error because celestial tend to have very huge numbers so we can't remove or replace the outliers with the mean so we will work with the outliers and use models that are robust for outliers like tree based models

In [37]:
print("spectral_type values: " ,train['spectral_type'].value_counts())
print()
print("galaxy_population values: " ,train['galaxy_population'].value_counts())
print()
print("class values: " ,train['class'].value_counts())

spectral_type values:  spectral_type
M      303323
A/F    122122
G/K    108546
O/B     43356
Name: count, dtype: int64

galaxy_population values:  galaxy_population
Red_Sequence    319565
Blue_Cloud      257782
Name: count, dtype: int64

class values:  class
GALAXY    377480
QSO       117143
STAR       82724
Name: count, dtype: int64


now we have 3 categorical data that we need to turn it numbers the model can learn from.  
Since the the categorical columns are nomial so I will use hot-key encoding


In [38]:
train = pd.get_dummies(train, columns=['spectral_type', 'galaxy_population'])
test = pd.get_dummies(test, columns=['spectral_type', 'galaxy_population'])


from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
train['class'] = le.fit_transform(train['class'])  # GALAXY=0, QSO=1, STAR=2


train = train.drop(columns=['id'])
test = test.drop(columns=['id'])

y_train = train['class']
X_train = train.drop(columns=['class'])

X_test = test

In [39]:
print(train.columns.tolist())

['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'class', 'spectral_type_A/F', 'spectral_type_G/K', 'spectral_type_M', 'spectral_type_O/B', 'galaxy_population_Blue_Cloud', 'galaxy_population_Red_Sequence']


In [ ]:
# Feature engineering color indices
for train in [X_train, X_test]:
    train['u_g'] = train['u'] - train['g']
    train['g_r'] = train['g'] - train['r']
    train['r_i'] = train['r'] - train['i']
    train['i_z'] = train['i'] - train['z']
    train['u_r'] = train['u'] - train['r']
    train['g_z'] = train['g'] - train['z']
for test in [X_train, X_test]:
    test['u_g'] = test['u'] - test['g']
    test['g_r'] = test['g'] - test['r']
    test['r_i'] = test['r'] - test['i']
    test['i_z'] = test['i'] - test['z']
    test['u_r'] = test['u'] - test['r']
    test['g_z'] = test['g'] - test['z']

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import RandomizedSearchCV



model = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    random_state=42,
    verbose=-1  # suppresses LightGBM's own logs so output is cleaner
)

param_grid = {
    'n_estimators': [300, 500, 700, 1000],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.001, 0.01, 0.05, 0.1],
    'subsample': [0.6, 0.7, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 1.0],
    'min_child_samples': [10, 20, 30],
    'reg_alpha': [0, 0.1, 0.5],       # L1 regularization (reduces overfitting)
    'reg_lambda': [0, 0.1, 0.5, 1.0]  # L2 regularization (reduces overfitting)
}

search = RandomizedSearchCV(
    model,
    param_distributions=param_grid,
    n_iter=50,          # increased from 20 to 50 for wider search
    scoring='accuracy',
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV accuracy:", search.best_score_)